In [35]:
import re, ast
import pandas as pd

with open("results/raw_shap_nightrun.md", "r", encoding="utf-8") as f:
    night_run = f.read()

In [24]:
all_blocks = night_run.split("Variable:")
all_blocks = [b.strip() for b in all_blocks if b.strip()]

all_blocks = all_blocks[1:]

In [25]:
all_blocks

["tradeflow_baci\\nRF basic: {'model_name': 'RandomForest_tradeflow_baci', 'n_train': 3640, 'n_test': 910, 'rmse': 2.1969234699638394, 'mae': 1.5707153376721754, 'r2': 0.48236444266050016, 'shap importance (grouped)': {'exporter_FE': np.float64(1.0825656624986466), 'importer_FE': np.float64(0.5285056627256377), 'other': {'contig': np.float64(0.3804110992038566), 'comlang_off': np.float64(0.10057214821250529), 'comlang_ethno': np.float64(0.062156051780437885), 'log_gdp_o': np.float64(0.7346613360759036), 'log_gdp_d': np.float64(0.4353649468565382), 'log_distw': np.float64(0.5444576907836274)}}}\\nRF full: {'model_name': 'RandomForest_tradeflow_baci', 'n_train': 3640, 'n_test': 910, 'rmse': 2.347837909776248, 'mae': 1.7718221272556067, 'r2': 0.40880539026556406, 'shap importance (grouped)': {'exporter_FE': np.float64(0.5906058946141746), 'importer_FE': np.float64(0.4119161509340978), 'other': {'contig': np.float64(0.3599775106497096), 'comlang_off': np.float64(0.07959070571625103), 'coml

In [28]:
df_names = ["lag0", "lag1", "lag2"]

results = []

for idx, block in enumerate(all_blocks):

    df_id = df_names[idx // 6]  # ✅ now safe

    # convert escaped newlines FIRST
    block = block.replace("\\n", "\n")

    lines = block.split("\n")
    target = lines[0].strip()

    for line in lines[1:]:
        if ":" not in line:
            continue

        name, dict_str = line.split(":", 1)

        # clean numpy floats
        dict_str = re.sub(r"np\.float(32|64)\((.*?)\)", r"\2", dict_str)

        try:
            data = ast.literal_eval(dict_str)

            results.append({
                "target": target,
                "model_spec": name.strip(),
                "data_version": df_id,
                **data
            })

        except Exception as e:
            print("FAILED:", name)


In [34]:
results

[{'target': 'tradeflow_baci',
  'model_spec': 'RF basic',
  'data_version': 'lag0',
  'model_name': 'RandomForest_tradeflow_baci',
  'n_train': 3640,
  'n_test': 910,
  'rmse': 2.1969234699638394,
  'mae': 1.5707153376721754,
  'r2': 0.48236444266050016,
  'shap importance (grouped)': {'exporter_FE': 1.0825656624986466,
   'importer_FE': 0.5285056627256377,
   'other': {'contig': 0.3804110992038566,
    'comlang_off': 0.10057214821250529,
    'comlang_ethno': 0.062156051780437885,
    'log_gdp_o': 0.7346613360759036,
    'log_gdp_d': 0.4353649468565382,
    'log_distw': 0.5444576907836274}}},
 {'target': 'tradeflow_baci',
  'model_spec': 'RF full',
  'data_version': 'lag0',
  'model_name': 'RandomForest_tradeflow_baci',
  'n_train': 3640,
  'n_test': 910,
  'rmse': 2.347837909776248,
  'mae': 1.7718221272556067,
  'r2': 0.40880539026556406,
  'shap importance (grouped)': {'exporter_FE': 0.5906058946141746,
   'importer_FE': 0.4119161509340978,
   'other': {'contig': 0.3599775106497096,

In [36]:
df = pd.DataFrame(results)
df[["model", "spec"]] = df["model_spec"].str.split(" ", expand=True)

In [37]:
df

,target,model_spec,data_version,model_name,n_train,n_test,rmse,mae,r2,shap importance (grouped),model,spec
0,tradeflow_baci,RF basic,lag0,RandomForest_tradeflow_baci,3640,910,2.196923,1.570715,0.482364,"{'exporter_FE': 1.0825656624986466, 'importer_...",RF,basic
1,tradeflow_baci,RF full,lag0,RandomForest_tradeflow_baci,3640,910,2.347838,1.771822,0.408805,"{'exporter_FE': 0.5906058946141746, 'importer_...",RF,full
2,tradeflow_baci,MLP basic,lag0,MLP_tradeflow_baci,3640,910,2.179484,1.634907,0.490550,"{'exporter_FE': 1.352158674525605, 'importer_F...",MLP,basic
3,tradeflow_baci,MLP full,lag0,MLP_tradeflow_baci,3640,910,12.632475,5.371200,-16.114773,"{'exporter_FE': 1.3345178550050958, 'importer_...",MLP,full
4,tradeflow_baci,XGB basic,lag0,XGBoost_tradeflow_baci,3640,910,2.248696,1.629406,0.457680,"{'exporter_FE': 1.1000466, 'importer_FE': 0.48...",XGB,basic
...,...,...,...,...,...,...,...,...,...,...,...,...
103,combined_trade_baci,RF full,lag2,RandomForest_combined_trade_baci,3276,910,1.639427,1.186319,0.542933,"{'exporter_FE': 0.3791631945410454, 'importer_...",RF,full
104,combined_trade_baci,MLP basic,lag2,MLP_combined_trade_baci,3276,910,1.591481,1.130970,0.569277,"{'exporter_FE': 0.9192257308366236, 'importer_...",MLP,basic
105,combined_trade_baci,MLP full,lag2,MLP_combined_trade_baci,3276,910,5.016347,3.091912,-3.279278,"{'exporter_FE': 0.8054708863387624, 'importer_...",MLP,full
106,combined_trade_baci,XGB basic,lag2,XGBoost_combined_trade_baci,3276,910,1.594707,1.107628,0.567529,"{'exporter_FE': 0.54252213, 'importer_FE': 0.5...",XGB,basic


In [38]:
df.sort_values("r2", ascending=False) \
  .groupby(["data_version", "target"]) \
  .head(1)[["data_version", "target", "model", "spec", "r2", "rmse"]]

,data_version,target,model,spec,r2,rmse
102,lag2,combined_trade_baci,RF,basic,0.596609,1.540159
66,lag1,combined_trade_baci,RF,basic,0.592371,1.548574
30,lag0,combined_trade_baci,RF,basic,0.583010,1.566150
18,lag0,tradeflow_imf_o,RF,basic,0.506164,2.079399
54,lag1,tradeflow_imf_o,RF,basic,0.501465,2.088969
90,lag2,tradeflow_imf_o,RF,basic,0.492830,2.106715
2,lag0,tradeflow_baci,MLP,basic,0.490550,2.179484
74,lag2,tradeflow_baci,MLP,basic,0.489421,2.181782
36,lag1,tradeflow_baci,RF,basic,0.478799,2.204550
6,lag0,tradeflow_comtrade_o,RF,basic,0.474438,1.927412


In [42]:
df[df["r2"] < 0][["data_version", "target", "model", "spec", "r2", "rmse"]]

,data_version,target,model,spec,r2,rmse
3,lag0,tradeflow_baci,MLP,full,-16.114773,12.632475
9,lag0,tradeflow_comtrade_o,MLP,full,-15.758658,10.883838
15,lag0,tradeflow_comtrade_d,MLP,full,-8.388824,8.516308
21,lag0,tradeflow_imf_o,MLP,full,-39.791952,18.898777
27,lag0,tradeflow_imf_d,MLP,full,-5.585960,7.502830
33,lag0,combined_trade_baci,MLP,full,-37.803703,15.108004
39,lag1,tradeflow_baci,MLP,full,-2.245709,5.501383
45,lag1,tradeflow_comtrade_o,MLP,full,-6.419370,7.239404
51,lag1,tradeflow_comtrade_d,MLP,full,-2.384678,5.112330
57,lag1,tradeflow_imf_o,MLP,full,-3.634321,6.369087


In [40]:
pivot = df.pivot_table(
    index=["data_version", "target", "model"],
    columns="spec",
    values="r2"
)

pivot["gain_full_vs_basic"] = pivot["full"] - pivot["basic"]

In [41]:
pivot

spec                                        basic       full  \
data_version target               model                        
lag0         combined_trade_baci  MLP    0.578084 -37.803703   
                                  RF     0.583010   0.504907   
                                  XGB    0.538384   0.511214   
             tradeflow_baci       MLP    0.490550 -16.114773   
                                  RF     0.482364   0.408805   
                                  XGB    0.457680   0.393614   
             tradeflow_comtrade_d MLP    0.390095  -8.388824   
                                  RF     0.401936   0.330716   
                                  XGB    0.408702   0.387649   
             tradeflow_comtrade_o MLP    0.433091 -15.758658   
                                  RF     0.474438   0.375135   
                                  XGB    0.470597   0.361553   
             tradeflow_imf_d      MLP    0.390011  -5.585960   
                                  RF     0.418554   0.324126   
                                  XGB    0.422088   0.372310   
             tradeflow_imf_o      MLP    0.408346 -39.791952   
                                  RF     0.506164   0.395427   
                                  XGB    0.490285   0.424648   
lag1         combined_trade_baci  MLP    0.552656  -7.868231   
                                  RF     0.592371   0.549592   
                                  XGB    0.551306   0.539300   
             tradeflow_baci       MLP    0.466705  -2.245709   
                                  RF     0.478799   0.453059   
                                  XGB    0.464905   0.448422   
             tradeflow_comtrade_d MLP    0.392139  -2.384678   
                                  RF     0.393451   0.370334   
                                  XGB    0.406880   0.393631   
             tradeflow_comtrade_o MLP    0.445549  -6.419370   
                                  RF     0.469270   0.394063   
                                  XGB    0.454143   0.450782   
             tradeflow_imf_d      MLP    0.354408  -1.874012   
                                  RF     0.417636   0.396411   
                                  XGB    0.416707   0.410892   
             tradeflow_imf_o      MLP    0.363036  -3.634321   
                                  RF     0.501465   0.465258   
                                  XGB    0.485147   0.484916   
lag2         combined_trade_baci  MLP    0.569277  -3.279278   
                                  RF     0.596609   0.542933   
                                  XGB    0.567529   0.537798   
             tradeflow_baci       MLP    0.489421  -1.896270   
                                  RF     0.470506   0.419169   
                                  XGB    0.468329   0.425391   
             tradeflow_comtrade_d MLP    0.434822  -2.215859   
                                  RF     0.405429   0.366226   
                                  XGB    0.406169   0.388852   
             tradeflow_comtrade_o MLP    0.444757  -2.421642   
                                  RF     0.470902   0.412220   
                                  XGB    0.461871   0.436728   
             tradeflow_imf_d      MLP    0.363979  -1.055524   
                                  RF     0.421711   0.373490   
                                  XGB    0.435797   0.369680   
             tradeflow_imf_o      MLP    0.379778  -1.562998   
                                  RF     0.492830   0.453689   
                                  XGB    0.479879   0.440943   

spec                                     gain_full_vs_basic  
data_version target               model                      
lag0         combined_trade_baci  MLP            -38.381786  
                                  RF              -0.078103  
                                  XGB             -0.027170  
             tradeflow_baci       MLP            -16.605323  
                                  RF       

In [43]:
def flatten_shap(shap_dict):
    flat = {}
    for k, v in shap_dict.items():
        if isinstance(v, dict):
            for sub_k, sub_v in v.items():
                flat[f"{k}_{sub_k}"] = float(sub_v)
        else:
            flat[k] = float(v)
    return flat

clean = []
for r in results:
    base = {k: v for k, v in r.items() if k != "shap importance (grouped)"}
    base.update(flatten_shap(r["shap importance (grouped)"]))
    clean.append(base)

df = pd.DataFrame(clean)

In [ ]:
df.columns = df.columns.str.replace("^other_", "", regex=True)
df.to_csv("results/clean_shap_results.csv", index=False)

In [52]:
df

,target,model_spec,data_version,model_name,n_train,n_test,rmse,mae,r2,exporter_FE,...,perpetrator_Rioters,perpetrator_State forces,target_Civilians,target_External/Other forces,target_Identity militia,target_Political militia,target_Protesters,target_Rebel group,target_Rioters,target_State forces
0,tradeflow_baci,RF basic,lag0,RandomForest_tradeflow_baci,3640,910,2.196923,1.570715,0.482364,1.082566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tradeflow_baci,RF full,lag0,RandomForest_tradeflow_baci,3640,910,2.347838,1.771822,0.408805,0.590606,...,0.041316,0.044154,0.032814,0.024855,0.051438,0.027322,0.020011,0.021321,0.020749,0.031437
2,tradeflow_baci,MLP basic,lag0,MLP_tradeflow_baci,3640,910,2.179484,1.634907,0.490550,1.352159,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,tradeflow_baci,MLP full,lag0,MLP_tradeflow_baci,3640,910,12.632475,5.371200,-16.114773,1.334518,...,0.176419,0.069155,0.030780,0.121707,0.047587,0.130834,0.039505,0.047563,0.067003,0.148676
4,tradeflow_baci,XGB basic,lag0,XGBoost_tradeflow_baci,3640,910,2.248696,1.629406,0.457680,1.100047,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,combined_trade_baci,RF full,lag2,RandomForest_combined_trade_baci,3276,910,1.639427,1.186319,0.542933,0.379163,...,0.027929,0.021419,0.023630,0.007062,0.026212,0.015602,0.023101,0.014722,0.010852,0.023351
104,combined_trade_baci,MLP basic,lag2,MLP_combined_trade_baci,3276,910,1.591481,1.130970,0.569277,0.919226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,combined_trade_baci,MLP full,lag2,MLP_combined_trade_baci,3276,910,5.016347,3.091912,-3.279278,0.805471,...,0.081049,0.084081,0.130330,0.050478,0.048266,0.037807,0.006965,0.024971,0.084320,0.105239
106,combined_trade_baci,XGB basic,lag2,XGBoost_combined_trade_baci,3276,910,1.594707,1.107628,0.567529,0.542522,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
df.groupby(["data_version", "model_name", "model_spec"])[
    ["other_log_gdp_o", "other_log_gdp_d"]
].mean()

other_log_gdp_o  \
data_version model_name                   model_spec                    
lag0         MLP_combined_trade_baci      MLP basic          0.670725   
                                          MLP full           0.406571   
             MLP_tradeflow_baci           MLP basic          0.389569   
                                          MLP full           0.391889   
             MLP_tradeflow_comtrade_d     MLP basic          0.371549   
...                                                               ...   
lag2         XGBoost_tradeflow_comtrade_o XGB full           0.726626   
             XGBoost_tradeflow_imf_d      XGB basic          0.905358   
                                          XGB full           0.999254   
             XGBoost_tradeflow_imf_o      XGB basic          0.925419   
                                          XGB full           0.958232   

                                                      other_log_gdp_d  
data_version model_name                   model_spec                   
lag0         MLP_combined_trade_baci      MLP basic          0.552173  
                                          MLP full           0.778606  
             MLP_tradeflow_baci           MLP basic          0.826830  
                                          MLP full           1.215230  
             MLP_tradeflow_comtrade_d     MLP basic          1.185636  
...                                                               ...  
lag2         XGBoost_tradeflow_comtrade_o XGB full           0.625343  
             XGBoost_tradeflow_imf_d      XGB basic          0.414626  
                                          XGB full           0.346113  
             XGBoost_tradeflow_imf_o      XGB basic          0.570763  
                                          XGB full           0.511560  

[108 rows x 2 columns]

In [63]:
df_filtered = df[(df["event_Riots"] > 0.1) & (df["model_spec"] != "MLP full")]

In [70]:
event_cols = df.columns[df.columns.str.startswith("disorder_")]

df_filtered = df[
    (df[event_cols].gt(0.1).any(axis=1)) &
    (df["model_spec"] != "MLP full")
]

In [71]:
df_filtered

,target,model_spec,data_version,model_name,n_train,n_test,rmse,mae,r2,exporter_FE,...,perpetrator_Rioters,perpetrator_State forces,target_Civilians,target_External/Other forces,target_Identity militia,target_Political militia,target_Protesters,target_Rebel group,target_Rioters,target_State forces
13,tradeflow_comtrade_d,RF full,lag0,RandomForest_tradeflow_comtrade_d,3640,910,2.273794,1.715931,0.330716,0.638055,...,0.039591,0.036367,0.029545,0.020093,0.049064,0.024860,0.047818,0.018554,0.031598,0.032613
47,tradeflow_comtrade_o,XGB full,lag1,XGBoost_tradeflow_comtrade_o,3458,910,1.969662,1.434321,0.450782,0.873813,...,0.050041,0.058144,0.031238,0.010955,0.029218,0.028960,0.009277,0.014139,0.043586,0.041297
55,tradeflow_imf_o,RF full,lag1,RandomForest_tradeflow_imf_o,3458,910,2.163496,1.602917,0.465258,0.718728,...,0.066597,0.030562,0.066453,0.009683,0.042244,0.021105,0.017160,0.027873,0.023098,0.034100
59,tradeflow_imf_o,XGB full,lag1,XGBoost_tradeflow_imf_o,3458,910,2.123358,1.528293,0.484916,0.965352,...,0.014556,0.068738,0.026946,0.014198,0.013673,0.016737,0.012281,0.025119,0.013866,0.032181
91,tradeflow_imf_o,RF full,lag2,RandomForest_tradeflow_imf_o,3276,910,2.186498,1.667785,0.453689,0.670995,...,0.077621,0.027566,0.045670,0.012134,0.040529,0.028316,0.027675,0.027464,0.019896,0.032755
